# Agente de Detección de Spam con IA
Este cuadernillo contiene el código completo para conectarse al correo, leer los mensajes no leídos, evaluarlos con Gemini y mostrar los resultados de forma estructurada.

**Nota:** Asegúrate de tener tu archivo `.env` configurado en esta misma carpeta.

In [ ]:
import imaplib
import email
import os
import json
from email.header import decode_header
import google.generativeai as genai
from dotenv import load_dotenv

class EmailAgent:
    def __init__(self, email_user, email_pass, gemini_api_key, imap_server="imap.gmail.com"):
        self.email_user = email_user
        self.email_pass = email_pass
        self.imap_server = imap_server
        self.mail = None
        genai.configure(api_key=gemini_api_key)
        self.model = genai.GenerativeModel('gemini-flash-latest')

    def connect(self):
        try:
            self.mail = imaplib.IMAP4_SSL(self.imap_server)
            self.mail.login(self.email_user, self.email_pass)
            print("Conectado exitosamente al correo.")
            return True
        except Exception as e:
            print(f"Error al conectar: {e}")
            return False

    def fetch_unread_emails(self, limit=5):
        self.mail.select("inbox")
        status, messages = self.mail.search(None, "UNSEEN")
        email_ids = messages[0].split()
        emails_data = []
        for e_id in email_ids[-limit:]:
            _, msg_data = self.mail.fetch(e_id, "(RFC822)")
            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_bytes(response_part[1])
                    subject, encoding = decode_header(msg["Subject"])[0]
                    if isinstance(subject, bytes):
                        try:
                            subject = subject.decode(encoding if encoding and encoding != "unknown-8bit" else "utf-8", errors="replace")
                        except LookupError:
                            subject = subject.decode("utf-8", errors="replace")
                    sender = msg.get("From")
                    body = self._get_email_body(msg)
                    emails_data.append({"id": e_id.decode() if isinstance(e_id, bytes) else e_id, "subject": subject, "sender": sender, "body": body})
        return emails_data

    def _get_email_body(self, msg):
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    try: return part.get_payload(decode=True).decode()
                    except: pass
        else:
            return msg.get_payload(decode=True).decode()
        return ""

    def process_emails(self, emails):
        results = []
        for em in emails:
            print(f"Procesando correo: {em['subject']}")
            analysis = self._analyze_with_gemini(em)
            if analysis.get("is_spam", False):
                print(f"[{em['subject']}] marcado como SPAM. Ignorando.")
                continue
            results.append({"email": em, "priority": analysis.get("priority", "Baja"), "suggested_reply": analysis.get("suggested_reply", "")})
        priority_map = {"Alta": 1, "Media": 2, "Baja": 3}
        results.sort(key=lambda x: priority_map.get(x["priority"], 3))
        return results

    def _analyze_with_gemini(self, email_data):
        prompt = f"""Actúa como un asistente ejecutivo inteligente. Analiza el siguiente correo electrónico:
        Remitente: {email_data['sender']}
        Asunto: {email_data['subject']}
        Cuerpo: {email_data['body']}
        Debes responder ÚNICAMENTE en formato JSON válido con la siguiente estructura (sin bloques de código extra):
        {{"is_spam": true o false, "priority": "Alta", "Media" o "Baja", "suggested_reply": "Un borrador de respuesta profesional en el idioma del correo, o vacío si es spam"}}"""
        try:
            response = self.model.generate_content(prompt)
            response_text = response.text.strip()
            if response_text.startswith("```json"):
                response_text = response_text.replace("```json", "").replace("```", "").strip()
            return json.loads(response_text)
        except Exception as e:
            print(f"Error consultando a Gemini: {e}")
            return {"is_spam": False, "priority": "Baja", "suggested_reply": ""}

# --- BLOQUE DE EJECUCIÓN ---
load_dotenv()
USER = os.getenv("EMAIL_USER")
PASS = os.getenv("EMAIL_PASS")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")

if USER and PASS and GEMINI_KEY:
    agent = EmailAgent(USER, PASS, GEMINI_KEY)
    if agent.connect():
        unread = agent.fetch_unread_emails(limit=5)
        if unread:
            processed = agent.process_emails(unread)
            with open("emails_procesados.json", "w", encoding="utf-8") as f:
                json.dump(processed, f, indent=2, ensure_ascii=False)
            print("\n¡Proceso terminado! Los resultados se han guardado en 'emails_procesados.json'.")
        else:
            print("No hay correos nuevos.")
else:
    print("Faltan credenciales en el archivo .env")


--- 
## Visualización de Resultados
Ejecuta la celda de abajo para leer el archivo JSON generado y mostrar los correos en un formato amigable visualmente utilizando HTML.

In [ ]:
import json
from IPython.display import display, HTML

try:
    with open("emails_procesados.json", "r", encoding="utf-8") as f:
        datos = json.load(f)
        
    if not datos:
        display(HTML("<h3>No hay correos para mostrar.</h3>"))
    else:
        # En Jupyter podemos renderizar tarjetas HTML para que se vea mucho más bonito que la terminal
        html_content = "<h2>📥 Bandeja de Entrada - Agente IA</h2><div style='display:flex; flex-direction:column; gap:15px;'>"
        
        color_map = {"Alta": "#ef4444", "Media": "#f59e0b", "Baja": "#10b981"}
        
        for item in datos:
            email = item.get("email", {})
            prio = item.get('priority', 'Baja')
            color = color_map.get(prio, "#888")
            
            cuerpo = email.get('body', '').strip()
            if len(cuerpo) > 300:
                cuerpo = cuerpo[:300].replace('\n', ' ') + " [...]"
            
            card = f"""
            <div style='border: 1px solid #e5e7eb; border-radius: 8px; padding: 15px; background-color: white; box-shadow: 0 1px 3px rgba(0,0,0,0.1);'>
                <div style='display:flex; justify-content:space-between; align-items:center;'>
                    <h3 style='margin:0; color:#111827; font-family:sans-serif;'>{email.get('subject', 'Sin Asunto')}</h3>
                    <span style='background-color:{color}; color:white; padding:4px 12px; border-radius:9999px; font-weight:600; font-size:0.8em; font-family:sans-serif;'>Prioridad {prio}</span>
                </div>
                <p style='margin:8px 0; color:#4b5563; font-family:sans-serif; font-size:0.95em;'><strong>De:</strong> {email.get('sender', 'Desconocido')}</p>
            """
            if item.get("suggested_reply"):
                card += f"<div style='background:#eff6ff; padding:12px; border-left:4px solid #3b82f6; margin:12px 0; border-radius:4px; font-family:sans-serif;'><strong style='color:#1d4ed8'>💡 Sugerencia de Respuesta:</strong><br><span style='color:#1e3a8a'>{item['suggested_reply']}</span></div>"
            
            card += f"<p style='color:#6b7280; font-size:0.9em; font-family:sans-serif; background:#f9fafb; padding:10px; border-radius:6px; font-style:italic;'><strong>Cuerpo:</strong><br>{cuerpo}</p></div>"
            html_content += card
            
        html_content += "</div>"
        display(HTML(html_content))
        
except FileNotFoundError:
    display(HTML("<h3 style='color:red;'>El archivo 'emails_procesados.json' no se encontró. Ejecuta la celda de arriba primero.</h3>"))
